# 28. Unsupervised Learning: Principal Component Analysis (PCA)

## Algorithm Category
**Type**: Unsupervised Learning - Dimensionality Reduction  
**Complexity**: Medium  
**Use Case**: Reduce dimensionality while preserving maximum variance

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand PCA and its mathematical foundation
- Implement PCA for dimensionality reduction
- Understand eigenvalues, eigenvectors, and explained variance
- Visualize principal components and data projections
- Determine optimal number of components
- Apply PCA for data preprocessing and visualization

## Historical Context

PCA was developed by Karl Pearson in 1901:
- Pearson, K. (1901): "On lines and planes of closest fit to systems of points in space"
- One of the oldest and most widely used dimensionality reduction techniques
- Foundation for many modern techniques

**Key Papers/References:**
- Pearson, K. (1901). "On lines and planes of closest fit to systems of points in space"
- Hotelling, H. (1933). "Analysis of a complex of statistical variables into principal components"

## When to Use Principal Component Analysis

PCA is appropriate when:
- You have high-dimensional data
- Features are correlated
- You want to reduce dimensionality for visualization
- You need to remove noise from data
- You want to reduce computational cost
- Features need to be decorrelated

## Theory & Mechanics

### Mathematical Foundation

PCA finds directions of maximum variance in data and projects data onto these directions.

**Covariance Matrix:**
$$C = \frac{1}{n-1} X^T X$$

Where $X$ is the centered data matrix.

**Eigenvalue Decomposition:**
$$C = P \Lambda P^T$$

Where:
- $P$: Matrix of eigenvectors (principal components)
- $\Lambda$: Diagonal matrix of eigenvalues (variances)

**Principal Components:**
- First PC: Direction of maximum variance
- Second PC: Direction of maximum variance orthogonal to first PC
- And so on...

**Projection:**
$$Y = X P_k$$

Where $P_k$ contains the first $k$ principal components.

**Explained Variance Ratio:**
$$\text{Explained Variance Ratio}_i = \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j}$$

### How It Works

1. **Center data**: Subtract mean from each feature
2. **Compute covariance matrix**: Calculate covariance between all feature pairs
3. **Eigenvalue decomposition**: Find eigenvectors and eigenvalues
4. **Select components**: Choose top k components based on explained variance
5. **Project data**: Transform data to lower-dimensional space

### Key Hyperparameters

- **n_components**: Number of components to keep
  - Integer: Exact number
  - Float (0-1): Keep components that explain this fraction of variance
  - 'mle': Use MLE to estimate number
- **whiten**: Whether to whiten the components (scale to unit variance)

### Advantages

- Reduces dimensionality while preserving variance
- Removes correlation between features
- Can improve model performance
- Helps with visualization
- Reduces overfitting risk
- Fast and efficient

### Limitations

- Assumes linear relationships
- May lose interpretability
- Sensitive to feature scaling
- May not capture non-linear patterns
- Components may not be meaningful


## Implementation

Let's implement PCA for dimensionality reduction.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Import our helper functions
from src.models.unsupervised import perform_pca

print("Libraries imported successfully!")


In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print(f"Original Dataset Shape: {X.shape}")
print(f"Features: {iris.feature_names}")

# Standardize features (important for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"\nPCA Results:")
print(f"  Reduced shape: {X_pca.shape}")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"  Total explained variance: {sum(pca.explained_variance_ratio_):.3f}")

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('Original Data (First 2 Features)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('PCA Projection (2D)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Explained Variance and Component Selection

Let's analyze explained variance to determine optimal number of components.


In [ ]:
# Fit PCA with all components to analyze variance
pca_full = PCA()
pca_full.fit(X_scaled)

# Calculate cumulative explained variance
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Explained Variance by Component')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'o-', markersize=8)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find number of components for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Number of components for 95% variance: {n_components_95}")
print(f"Explained variance with {n_components_95} components: {cumulative_variance[n_components_95-1]:.3f}")

# Print explained variance for each component
print("\nExplained Variance by Component:")
for i, var in enumerate(explained_variance):
    print(f"  PC{i+1}: {var:.3f} ({var*100:.1f}%)")


## Principal Component Analysis

Let's examine the principal components and their relationship to original features.


In [ ]:
# Get principal components (eigenvectors)
components = pca.components_

# Create heatmap of component loadings
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(components, cmap='coolwarm', aspect='auto')
ax.set_xticks(range(len(iris.feature_names)))
ax.set_xticklabels(iris.feature_names, rotation=45, ha='right')
ax.set_yticks(range(2))
ax.set_yticklabels([f'PC{i+1}' for i in range(2)])
ax.set_title('Principal Component Loadings')
plt.colorbar(im, ax=ax, label='Loading')
plt.tight_layout()
plt.show()

# Print component loadings
print("Principal Component Loadings:")
for i, pc in enumerate(components):
    print(f"\nPC{i+1}:")
    for j, feature in enumerate(iris.feature_names):
        print(f"  {feature}: {pc[j]:.3f}")


## Validation & Testing

Let's validate PCA and test reconstruction error.


In [ ]:
# Test reconstruction
X_reconstructed = pca.inverse_transform(X_pca)

# Calculate reconstruction error
reconstruction_error = np.mean((X_scaled - X_reconstructed) ** 2)
print(f"Reconstruction Error (MSE): {reconstruction_error:.6f}")

# Compare original vs reconstructed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
axes[0].set_xlabel('Feature 1 (standardized)')
axes[0].set_ylabel('Feature 2 (standardized)')
axes[0].set_title('Original Data')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
axes[1].set_xlabel('Feature 1 (reconstructed)')
axes[1].set_ylabel('Feature 2 (reconstructed)')
axes[1].set_title('Reconstructed Data (from 2 PCs)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Assertions
assert X_pca.shape[1] == 2, "Should have 2 principal components"
assert sum(pca.explained_variance_ratio_) > 0.9, "Should explain at least 90% variance"
print("\n✓ Validation checks passed")


## Real-World Application

Let's apply PCA to a higher-dimensional dataset.


In [ ]:
# Load Wine dataset (higher dimensional)
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print(f"Wine Dataset Shape: {X_wine.shape}")
print(f"Number of features: {X_wine.shape[1]}")

# Standardize
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

# Apply PCA
pca_wine = PCA(n_components=0.95)  # Keep 95% of variance
X_wine_pca = pca_wine.fit_transform(X_wine_scaled)

print(f"\nPCA Results:")
print(f"  Reduced shape: {X_wine_pca.shape}")
print(f"  Number of components: {pca_wine.n_components_}")
print(f"  Explained variance: {sum(pca_wine.explained_variance_ratio_):.3f}")

# Visualize first 2 components
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_wine_pca[:, 0], X_wine_pca[:, 1], c=y_wine, 
                      cmap='viridis', s=50, alpha=0.7)
plt.xlabel(f'PC1 ({pca_wine.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_wine.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Wine Dataset: PCA Projection (2D)')
plt.colorbar(scatter, label='Class')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compare dimensionality reduction
print(f"\nDimensionality Reduction:")
print(f"  Original: {X_wine.shape[1]} features")
print(f"  Reduced: {pca_wine.n_components_} components")
print(f"  Reduction: {(1 - pca_wine.n_components_/X_wine.shape[1])*100:.1f}%")


## Summary & Key Takeaways

### Key Concepts Learned

1. **PCA Basics**
   - Linear dimensionality reduction technique
   - Finds directions of maximum variance
   - Projects data onto principal components
   - Decorrelates features

2. **Mathematical Foundation**
   - Eigenvalue decomposition of covariance matrix
   - Principal components are eigenvectors
   - Explained variance from eigenvalues
   - Orthogonal components

3. **Component Selection**
   - Use explained variance ratio
   - Common thresholds: 95% or 99% variance
   - Scree plot to visualize variance
   - Elbow method for selection

4. **Best Practices**
   - Always standardize features before PCA
   - Visualize explained variance
   - Consider interpretability vs dimensionality
   - Use for preprocessing before ML models
   - Can improve model performance

### When to Use Principal Component Analysis

✅ **Good for:**
- High-dimensional data
- Correlated features
- Data visualization (reduce to 2D/3D)
- Noise reduction
- Feature decorrelation
- Reducing computational cost
- Preprocessing for ML models

❌ **Not ideal for:**
- Non-linear relationships (use t-SNE, UMAP)
- When interpretability is crucial
- When all features are important
- Very sparse data
- When features are already uncorrelated

### Next Steps

- Compare with **t-SNE** for non-linear dimensionality reduction
- Use **Kernel PCA** for non-linear relationships
- Apply **Incremental PCA** for large datasets
- Explore **Sparse PCA** for feature selection
- Use PCA for **anomaly detection** (reconstruction error)
